# Модуль 3. Не пересчитываем дважды: кеширование с `Memory`

## Зачем этот модуль

Представьте, что вы написали функцию, которая:
- читает 10 гигабайт данных из базы;
- чистит пропуски;
- создаёт 50 новых признаков;
- обучает на этих данных модель.

Весь процесс занимает 30 минут. Вы запустили его, пошли пить кофе, вернулись — всё готово. Но теперь вы хотите поменять один гиперпараметр модели. Приходится заново ждать 30 минут предобработки, хотя данные-то не изменились.

Или другой пример: ваш backend-приложение каждые 5 минут делает тяжёлый запрос к внешнему API. Ответ от API редко меняется, но вы всё равно ждёте по 10 секунд на каждый запрос.

**Кеширование** решает эту проблему: если функция вызвана с теми же аргументами — вернуть готовый результат из памяти или с диска, не выполняя вычисления заново.

В этом модуле мы научимся использовать `joblib.Memory` — инструмент, который делает это автоматически.

## 1. Что такое кеширование (мемоизация)

### Простая аналогия

Вы — учитель математики. Ученик спрашивает: «Сколько будет 127 × 349?» Вы считаете в столбик, отвечаете: «44 323», и записываете ответ в тетрадь. Через 5 минут тот же ученик спрашивает снова. Вы не считаете заново — просто смотрите в тетрадь и называете ответ.

**Тетрадь — это кеш.** Если вопрос уже был — ответ берётся из тетради. Если вопрос новый — решаем и записываем.

### Техническое определение

**Мемоизация** (memoization) — техника оптимизации, при которой результат выполнения функции сохраняется, и при повторном вызове с теми же аргументами возвращается сохранённый результат вместо повторного вычисления.

Важно: `joblib.Memory` кеширует результаты **на диске**, а не только в оперативной памяти. Это значит, что если вы перезапустите компьютер — кеш не пропадёт.

### Когда кеширование полезно

| Сценарий | Почему кеш помогает |
|----------|-------------------|
| Предобработка данных | Данные меняются редко, обработка долгая |
| Запросы к внешним API | Ответы кешируются на время, экономятся запросы и деньги |
| Feature engineering | Создание сложных признаков занимает минуты |
| Обучение с перебором гиперпараметров | Общие шаги пайплайна не пересчитываются |
| Загрузка и парсинг больших файлов | Файл прочитан один раз, потом берётся из кеша |

### Когда кеширование вредно

- Функция выполняется за миллисекунды — накладные расходы на чтение с диска будут дольше вычисления.
- Аргументы функции огромные — их хеширование займёт больше времени, чем сама функция.
- Результат зависит от внешнего состояния (текущее время, случайные числа без `random_state`) — кеш вернёт устаревший результат.

## 2. Создание объекта `Memory`

### Синтаксис

In [ ]:
from joblib import Memory

# Указываем папку, где будут храниться закешированные результаты
memory = Memory(location='./cache', verbose=0)

| Параметр | Что делает |
|----------|-----------|
| `location` | Путь к папке на диске, где хранится кеш. Если папки нет — `joblib` создаст её |
| `verbose` | Уровень «болтливости». `0` — молчит, `1` — пишет, когда использует кеш, `2` — подробный вывод |

### Что происходит при создании

`joblib` создаёт папку `./cache` (если её нет) и готовится отслеживать функции. Сам по себе объект `memory` пока ничего не делает — он ждёт, когда вы повесите на функцию декоратор.

## 3. Декоратор `@memory.cache`

### Как это работает

In [ ]:
from joblib import Memory
import time

memory = Memory(location='./my_cache', verbose=1)

@memory.cache
def slow_sum(a, b):
    time.sleep(2)  # имитация долгой работы
    return a + b

# Первый вызов — считает 2 секунды
result1 = slow_sum(10, 20)
print(result1)  # 30

# Второй вызов с теми же аргументами — мгновенно из кеша
result2 = slow_sum(10, 20)
print(result2)  # 30

При первом вызове `slow_sum(10, 20)`:
1. `joblib` вычисляет **хеш** аргументов `(10, 20)` + хеш исходного кода функции.
2. Проверяет папку `./my_cache` — такого хеша там нет.
3. Выполняет функцию `slow_sum`.
4. Сохраняет результат (`30`) в папку `./my_cache` под именем, основанным на хеше.

При втором вызове `slow_sum(10, 20)`:
1. `joblib` снова вычисляет хеш `(10, 20)` + код функции.
2. Находит в `./my_cache` файл с таким хешем.
3. Загружает результат (`30`) с диска, не выполняя функцию.
4. Возвращает результат мгновенно.

### Что видно в папке кеша

После первого запуска в папке `./my_cache` появится структура примерно такая:

In [ ]:
my_cache/
└── joblib/
    └── __main__/
        └── slow_sum/
            └── 3a7f.../
                ├── func_code.py         # сохранённый исходный код
                ├── output.pkl           # результат функции
                └── metadata.json        # информация о вызове

`joblib` организует файлы так, чтобы быстро находить нужный результат по хешу.

## 4. Как `joblib` хеширует аргументы и код

### Хеширование аргументов

`joblib` превращает аргументы функции в уникальную «отпечаток» — хеш. Если аргументы изменились хоть на один бит — хеш изменится, и функция пересчитается.

In [ ]:
@memory.cache
def process(data, multiplier):
    return data * multiplier

# Разные аргументы = разные хеши = разные кеш-записи
process(100, 2)  # кешируется результат 200
process(100, 3)  # кешируется результат 300 (новая запись)
process(100, 2)  # берётся из кеша — 200

### Хеширование исходного кода

Важная особенность: `joblib` хеширует не только аргументы, но и **тело функции**. Если вы измените код функции — старый кеш станет недействительным.

In [ ]:
@memory.cache
def greet(name):
    return f"Hello, {name}!"

greet("Alice")  # кешируется

# Теперь меняем функцию
@memory.cache
def greet(name):
    return f"Hi there, {name}!"  # код изменился!

greet("Alice")  # пересчитается, потому что хеш кода изменился

Это защищает от ошибки: вы не получите старый результат от новой функции.

### Что можно передавать в аргументах

`joblib` умеет хешировать:
- числа, строки, булевы значения;
- списки, кортежи, словари (включая вложенные);
- `numpy`-массивы;
- `pandas.DataFrame` (но это медленно — лучше передавать путь к файлу).

Не рекомендуется передавать:
- функции как аргументы (хеширование сложное и ненадёжное);
- объекты с изменяемым состоянием;
- огромные массивы напрямую — лучше передавать путь к файлу.

## 5. Параметры `Memory`

### `verbose`: контроль вывода

In [ ]:
memory = Memory(location='./cache', verbose=0)   # тишина
memory = Memory(location='./cache', verbose=1)   # "Using cached result..." 
memory = Memory(location='./cache', verbose=2)   # подробности о хешировании

При `verbose=1` вы увидите в консоли:

In [ ]:
[Memory] Calling __main__--...-slow_sum...
slow_sum - 2.0s, 2.0min
[Memory] Calling __main__--...-slow_sum...
slow_sum - 0.0s, 0.0min  (cached)

### `bytes_limit`: ограничение размера кеша

In [ ]:
memory = Memory(location='./cache', bytes_limit=1024 * 1024 * 100)  # 100 МБ

Если кеш превысит 100 МБ, `joblib` начнёт удалять самые старые записи (LRU — Least Recently Used).

### `compress`: сжатие кеша

In [ ]:
memory = Memory(location='./cache', compress=3)

Результаты функций будут сохраняться со сжатием — полезно, если функция возвращает большие массивы.

### `mmap_mode`: memory mapping для кеша

In [ ]:
memory = Memory(location='./cache', mmap_mode='r')

Если функция возвращает большие `numpy`-массивы, они будут загружаться через memory mapping.

## 6. Инвалидация кеша: когда нужно очистить

### Ручная очистка всего кеша

In [ ]:
memory.clear(warn=False)

Удаляет **все** закешированные результаты. Полезно, если вы знаете, что данные устарели.

### Очистка конкретной функции

In [ ]:
# Доступ к кешу конкретной функции
slow_sum.cache.clear()

### Автоматическая инвалидация

Кеш становится недействительным автоматически, если:
- изменились аргументы функции;
- изменился исходный код функции;
- изменились зависимости функции (но `joblib` этого не отследит!).

> **Важно:** `joblib` не отслеживает изменения **внешних файлов**. Если ваша функция читает `data.csv`, и вы изменили этот файл — `joblib` об этом не узнает и вернёт старый кешированный результат.

### Решение для внешних файлов

Передавайте **путь к файлу** как аргумент, а не читайте файл внутри функции вне декоратора:

In [ ]:
# ПЛОХО: joblib не увидит изменение файла
data = pd.read_csv('data.csv')

@memory.cache
def process():
    return data.mean()  # всегда будет кешироваться, даже если CSV изменился

# ХОРОШО: путь — часть аргументов, joblib хеширует строку-путь
@memory.cache
def process(file_path):
    data = pd.read_csv(file_path)
    return data.mean()

process('data.csv')  # если файл изменится — пересчитает (но только если вы измените имя или добавите аргумент)

> **На самом деле:** `joblib` хеширует строку `'data.csv'`, а не содержимое файла. Если вы перезаписали файл, но имя осталось тем же — `joblib` **не пересчитает**. Для отслеживания содержимого файла нужно дополнительно хешировать файл самостоятельно или использовать `mtime` (время модификации).

## 7. Backend-сценарии

### Сценарий A: Кеширование предобработки в API

In [ ]:
from joblib import Memory
from fastapi import FastAPI
import pandas as pd

memory = Memory(location='./api_cache', verbose=1)

@memory.cache
def expensive_feature_engineering(raw_data_path):
    """Долгая предобработка данных"""
    df = pd.read_csv(raw_data_path)
    df = create_features(df)  # сложные вычисления
    df = apply_transformations(df)
    return df

app = FastAPI()

@app.post("/train")
def train_model(data_path: str):
    # Если данные уже обрабатывались — мгновенно из кеша
    processed = expensive_feature_engineering(data_path)
    model = train_on(processed)
    return {"status": "trained"}

### Сценарий B: Кеширование запросов к внешнему API

In [ ]:
import requests
from joblib import Memory

memory = Memory(location='./external_api_cache', verbose=1)

@memory.cache
def fetch_exchange_rate(currency: str):
    """Дорогой запрос к API курсов валют"""
    response = requests.get(f"https://api.example.com/rate/{currency}")
    return response.json()

# Первый вызов — 2 секунды (сетевой запрос)
rate1 = fetch_exchange_rate("USD")

# Второй вызов — мгновенно из кеша
rate2 = fetch_exchange_rate("USD")

### Сценарий C: Кеширование в ML-экспериментах

In [ ]:
from joblib import Memory

memory = Memory(location='./ml_cache', verbose=1)

@memory.cache
def load_and_clean_data(path):
    df = pd.read_csv(path)
    df = df.dropna()
    df = df[df['age'] > 0]
    return df

@memory.cache
def create_features(df):
    df['income_per_age'] = df['income'] / df['age']
    df['is_premium'] = df['spending'] > 1000
    return df

# Весь ETL закеширован
raw = load_and_clean_data('customers.csv')
features = create_features(raw)

# При повторном запуске — мгновенно, если данные не менялись

## 8. Практика: задания

### Задание 3.1: «Первый кеш»

1. Создайте объект `Memory` с папкой `./lesson_cache`.
2. Напишите функцию `slow_multiply(a, b)`, которая:
   - ждёт 3 секунды (`time.sleep(3)`);
   - возвращает `a * b`.
3. Повесьте на неё декоратор `@memory.cache`.
4. Вызовите `slow_multiply(7, 8)` дважды подряд.
5. Убедитесь, что первый вызов занял ~3 секунды, а второй — мгновенно.
6. Посмотрите, что появилось в папке `./lesson_cache`.

### Задание 3.2: «Кеш и изменение аргументов»

1. Используйте функцию из задания 3.1.
2. Вызовите её с аргументами `(7, 8)`, `(7, 9)`, `(10, 8)`.
3. Проверьте, сколько подпапок появилось в кеше (должно быть 3).
4. Вызовите `(7, 8)` снова — убедитесь, что берётся из кеша.

### Задание 3.3: «Инвалидация при изменении кода»

1. Создайте функцию `greet(name)`, возвращающую `f"Hello, {name}"`, с кешем.
2. Вызовите `greet("World")` — закешируется.
3. Измените функцию, чтобы она возвращала `f"Hi, {name}"`.
4. Вызовите `greet("World")` снова — убедитесь, что результат пересчитался (код изменился).
5. Очистите кеш через `memory.clear(warn=False)`.

### Задание 3.4: «Кеш в ML-контексте»

1. Напишите функцию `prepare_data(file_path)`, которая:
   - читает CSV (можно использовать `make_classification` + `to_csv` для создания файла);
   - масштабирует признаки через `StandardScaler`;
   - возвращает масштабированный `numpy`-массив.
2. Закешируйте её через `@memory.cache`.
3. Вызовите дважды с одним и тем же путём — замерьте время.
4. Удалите файл CSV, создайте новый с тем же именем (другие данные).
5. Вызовите функцию снова — что произошло? Почему?

## 9. Эталонное решение

### Решение 3.1

In [ ]:
from joblib import Memory
import time

memory = Memory(location='./lesson_cache', verbose=1)

@memory.cache
def slow_multiply(a, b):
    time.sleep(3)
    return a * b

# Первый вызов
print("Первый вызов...")
result1 = slow_multiply(7, 8)
print(f"Результат: {result1}")

# Второй вызов — мгновенно
print("Второй вызов...")
result2 = slow_multiply(7, 8)
print(f"Результат: {result2}")

**Ожидаемый вывод:**

In [ ]:
Первый вызов...
[Memory] Calling __main__--...-slow_multiply...
slow_multiply - 3.0s, 0.1min
Результат: 56
Второй вызов...
[Memory] Calling __main__--...-slow_multiply...
slow_multiply - 0.0s, 0.0min  (cached)
Результат: 56

### Решение 3.4 (ML-контекст)

In [ ]:
from joblib import Memory
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
import time

# Создаём тестовый CSV
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
df = pd.DataFrame(X, columns=[f'f{i}' for i in range(10)])
df.to_csv('test_data.csv', index=False)

memory = Memory(location='./ml_cache', verbose=1)

@memory.cache
def prepare_data(file_path):
    df = pd.read_csv(file_path)
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df)
    return scaled

# Первый вызов
t0 = time.time()
data1 = prepare_data('test_data.csv')
print(f"Первый вызов: {time.time() - t0:.3f} сек")

# Второй вызов — из кеша
t0 = time.time()
data2 = prepare_data('test_data.csv')
print(f"Второй вызов: {time.time() - t0:.3f} сек")

# Создаём новый файл с тем же именем
X_new, _ = make_classification(n_samples=1000, n_features=10, random_state=99)
df_new = pd.DataFrame(X_new, columns=[f'f{i}' for i in range(10)])
df_new.to_csv('test_data.csv', index=False)

# Третий вызов — joblib НЕ узнает, что файл изменился!
t0 = time.time()
data3 = prepare_data('test_data.csv')
print(f"Третий вызов (файл изменён!): {time.time() - t0:.3f} сек — взято из кеша!")
print("ПРОБЛЕМА: joblib хеширует строку-путь, а не содержимое файла")

## 10. Вопросы для самопроверки

1. **Что такое мемоизация простыми словами?**  
   *(Ответ: сохранение результата функции, чтобы при повторном вызове с теми же аргументами не считать заново.)*

2. **Где хранятся результаты `joblib.Memory` — в RAM или на диске?**  
   *(Ответ: на диске, в указанной папке. Это позволяет сохранять кеш между перезапусками программы.)*

3. **Что хеширует `joblib` для проверки кеша?**  
   *(Ответ: аргументы функции и исходный код функции. Если что-то из этого изменилось — кеш инвалидируется.)*

4. **Почему `joblib` может вернуть устаревший результат, если мы изменили CSV-файл, но оставили то же имя файла?**  
   *(Ответ: потому что `joblib` хеширует строку с путём, а не содержимое файла. Он не знает, что файл на диске изменился.)*

5. **Как полностью очистить кеш?**  
   *(Ответ: вызвать `memory.clear(warn=False)`.)*

6. **Когда кеширование бесполезно или вредно?**  
   *(Ответ: когда функция выполняется очень быстро (меньше времени, чем чтение с диска), или когда результат зависит от внешнего состояния, которое `joblib` не отслеживает.)*

## Итоги модуля

- **`joblib.Memory`** — это механизм кеширования результатов функций на диске.
- Декоратор `@memory.cache` автоматически сохраняет результат при первом вызове и возвращает его из кеша при повторных.
- `joblib` хеширует **аргументы** и **исходный код** функции. Изменение любого из них инвалидирует кеш.
- Кеш хранится в папке на диске и переживает перезапуск программы.
- `joblib` **не отслеживает внешние файлы** — будьте осторожны с функциями, читающими файлы по пути.
- В backend кеширование ускоряет ответы API, экономит запросы к внешним сервисам и ускоряет ML-пайплайны.

**В следующем модуле** мы изучим третий инструмент `joblib` — **`Parallel` и `delayed`**, которые позволяют распараллеливать циклы и ускорять вычисления в несколько раз.